In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [2]:
# ---------------------------------------------------------
# 1. 데이터 로드 및 전처리
# ---------------------------------------------------------
class PowerDataset(Dataset):
    def __init__(self, data, window_size=24):
        # 입력 데이터가 Series나 Array일 수 있으므로 변환
        self.data = torch.FloatTensor(np.array(data))
        self.window_size = window_size

    def __len__(self):
        return len(self.data) - self.window_size

    def __getitem__(self, idx):
        # 24시간 데이터를 보고 1시간 뒤(25번째)를 예측
        x = self.data[idx : idx + self.window_size]
        y = self.data[idx + self.window_size]
        # 모델 출력이 (batch, 1) 형태이므로 y도 동일하게 view
        return x, y.view(1)

def prepare_data(file_path):
    df = pd.read_csv(file_path)
    df['mrdDt'] = pd.to_datetime(df['mrdDt'], format='%Y-%m-%d %H')
    df = df.sort_values('mrdDt')
    
    values = df['pwrQrt'] 
    
    # 데이터 분할 (10:1:1)
    train_size = int(len(values) * (10/12))
    val_size = int(len(values) * (1/12))
    
    train_raw = values[:train_size]
    val_raw = values[train_size : train_size+val_size]
    test_raw = values[train_size+val_size:]

    # [통일 핵심] Train 데이터의 Max값 찾기
    P_MAX = train_raw.max()
    
    # 모든 데이터를 이 P_MAX로 나누기 (Scaling)
    train_data = train_raw / P_MAX
    val_data = val_raw / P_MAX
    test_data = test_raw / P_MAX
    
    return train_data, val_data, test_data, P_MAX

# --- 실행부 ---
FILE_PATH = '1510080003.csv'
train_data, val_data, test_data, P_MAX = prepare_data(FILE_PATH)

# Dataset 생성 (window_size=24)
train_dataset = PowerDataset(train_data, window_size=24)
val_dataset = PowerDataset(val_data, window_size=24)
test_dataset = PowerDataset(test_data, window_size=24)

# DataLoader 설정 (batch_size는 훈련 효율을 위해 32 유지)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [3]:
# ---------------------------------------------------------
# 2. 모델 정의
# ---------------------------------------------------------
class MLPModel(nn.Module):
    def __init__(self, input_dim=24):
        super(MLPModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

In [4]:
# ---------------------------------------------------------
# 3. 모델 훈련 및 저장
# ---------------------------------------------------------
SEQ_LENGTH = 24
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLPModel(input_dim=SEQ_LENGTH).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 100
best_val_mae = float('inf')

for epoch in range(EPOCHS):
    # --- 1. Training Phase (학습은 여전히 MSE로 진행) ---
    model.train()
    train_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets) # MSE Loss
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # --- 2. Validation Phase (MAE * P_MAX로 변경) ---
    model.eval()
    val_mae_acc = 0
    val_count = 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            # |예측 - 실제|의 합계를 구함
            val_mae_acc += torch.abs(outputs - targets).sum().item()
            val_count += targets.size(0)
    
    # 실제 단위 MAE 계산: (누적 오차 / 샘플 수) * P_MAX
    avg_val_mae = (val_mae_acc / val_count) * P_MAX
            
    # --- 3. Test Phase (MAE * P_MAX로 변경) ---
    test_mae_acc = 0
    test_count = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            test_mae_acc += torch.abs(outputs - targets).sum().item()
            test_count += targets.size(0)
    
    avg_test_mae = (test_mae_acc / test_count) * P_MAX
    
    avg_train_loss = train_loss / len(train_loader) # 이건 훈련용 MSE
    
    # Best Model Save (기준을 실제 단위 MAE로 변경)
    if avg_val_mae < best_val_mae:
        best_val_mae = avg_val_mae
        torch.save(model.state_dict(), 'mlp_best.pt') # 또는 'transformer_best.pt'
        print(f"Epoch {epoch+1}: Model saved (Val MAE: {avg_val_mae:.6f} kW | Test MAE: {avg_test_mae:.6f} kW)")

    # 10 에포크마다 진행 상황 출력
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}]")
        print(f"Train MSE: {avg_train_loss:.6f}")
        print(f"Val MAE  : {avg_val_mae:.6f} kW")
        print(f"Test MAE : {avg_test_mae:.6f} kW")
        print("-" * 50)

print(f"Training Complete. Best Val MAE: {best_val_mae:.6f} kW")

Epoch 1: Model saved (Val MAE: 0.158863 kW | Test MAE: 0.192948 kW)
Epoch 2: Model saved (Val MAE: 0.157982 kW | Test MAE: 0.188460 kW)
Epoch 3: Model saved (Val MAE: 0.150211 kW | Test MAE: 0.181493 kW)
Epoch 6: Model saved (Val MAE: 0.149459 kW | Test MAE: 0.180322 kW)
Epoch 7: Model saved (Val MAE: 0.147961 kW | Test MAE: 0.177068 kW)
Epoch [10/100]
Train MSE: 0.003384
Val MAE  : 0.155887 kW
Test MAE : 0.184828 kW
--------------------------------------------------
Epoch 11: Model saved (Val MAE: 0.146587 kW | Test MAE: 0.176825 kW)
Epoch [20/100]
Train MSE: 0.003265
Val MAE  : 0.148605 kW
Test MAE : 0.178911 kW
--------------------------------------------------
Epoch [30/100]
Train MSE: 0.003229
Val MAE  : 0.151960 kW
Test MAE : 0.185332 kW
--------------------------------------------------
Epoch [40/100]
Train MSE: 0.003127
Val MAE  : 0.151027 kW
Test MAE : 0.187465 kW
--------------------------------------------------
Epoch [50/100]
Train MSE: 0.003068
Val MAE  : 0.155273 kW
Test 